# Corn

In [1]:
"""
Optimized RNN Yield Regressor  (FP32 — no AMP)
- Mirrors the LSTM script exactly: same data pipeline, metrics, fold structure
- FP32 throughout (avoids cuBLAS FP16 issues)
- Persistent DataLoader workers + pinned memory
- Early stopping (patience=5)
- Cosine LR annealing
- All metrics: R², RMSE, MAE, NRMSE, MAPE, Zone Accuracy + confusion matrix
- Per-fold + aggregate timing with ETA
"""

import re, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  CONFIG  ← only section you need to edit
# ─────────────────────────────────────────────
DATA_PATH   = "../data/corn_2016_2023.parquet"   # swap crop here
TARGET_COL  = "yield"
GROUP_COL   = "farm_name"
N_SPLITS    = 5
N_EPOCHS    = 30         # RNN converges faster than LSTM; 30 is a safe ceiling
PATIENCE    = 5          # stop early if val loss plateaus
BATCH_SIZE  = 16_384
LR          = 3e-4       # slightly lower than LSTM — RNN gradients can be noisy
HIDDEN      = 64         # RNN is simpler than LSTM; 64 is plenty
NUM_WORKERS = 8
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────
# 1.  LOAD & CLEAN
# ─────────────────────────────────────────────
t0_total = time.time()
print("=" * 60)
print("RNN Yield Regressor  —  FP32, no AMP")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})"
      if torch.cuda.is_available() else ""))
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df = df[df[TARGET_COL].notna()].copy()

drop_cols  = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]
drop_cols += [c for c in df.columns
              if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]
df.drop(columns=drop_cols, errors="ignore", inplace=True)
print(f"Loaded  : {len(df):,} rows | {len(df.columns)} cols  "
      f"(dropped {len(drop_cols)} cols)")

# ─────────────────────────────────────────────
# 2.  BUILD FEATURES
# ─────────────────────────────────────────────
def _suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_map = {_suffix(c): c for c in df.columns if c.startswith("VV_") and _suffix(c)}
vh_map = {_suffix(c): c for c in df.columns if c.startswith("VH_") and _suffix(c)}
common = sorted(set(vv_map) & set(vh_map))
T      = len(common)

vv_arr = df[[vv_map[s] for s in common]].to_numpy(dtype=np.float32)
vh_arr = df[[vh_map[s] for s in common]].to_numpy(dtype=np.float32)
X_seq  = np.stack([vv_arr, vh_arr], axis=2)                              # (N, T, 2)

nan_mask     = np.isnan(X_seq).astype(np.float32)
X_seq_filled = np.concatenate([np.nan_to_num(X_seq), nan_mask], axis=2)  # (N, T, 4)

STATIC_COLS = ["Year", "latitude"]
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)

y_raw  = df[TARGET_COL].to_numpy(dtype=np.float32)
groups = df[GROUP_COL].astype(str).to_numpy()
years  = df["Year"].to_numpy(dtype=np.int32)

print(f"X_seq   : {X_seq_filled.shape}  (T={T} steps, 4 ch incl. NaN mask)")
print(f"X_static: {X_static.shape}")
print(f"NaNs after fill: {np.isnan(X_seq_filled).sum()}")

gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y_raw, groups=groups))
print(f"Folds   : {len(folds)}  (GroupKFold on '{GROUP_COL}')\n")

# ─────────────────────────────────────────────
# 3.  MODEL
# ─────────────────────────────────────────────
class RNNRegressor(nn.Module):
    def __init__(self, seq_dim: int, stat_dim: int, hidden: int = 64):
        super().__init__()
        self.rnn = nn.RNN(seq_dim, hidden, num_layers=2,
                          batch_first=True, dropout=0.2, nonlinearity="tanh")
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),       nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        _, h = self.rnn(x_seq)            # h: (num_layers, B, hidden)
        s    = self.stat_mlp(x_stat)
        return self.head(torch.cat([h[-1], s], dim=1)).squeeze(1)

# ─────────────────────────────────────────────
# 4.  METRIC HELPERS
# ─────────────────────────────────────────────
def fmt_time(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m:02d}m {s:02d}s"

def all_metrics(y_true, y_pred, yr_true_ratio, yr_pred_ratio, tag=""):
    r2    = r2_score(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    rng   = float(np.max(y_true) - np.min(y_true))
    nrmse = (rmse / rng * 100) if rng > 0 else 0.0
    mape  = float(np.mean(np.abs((y_true - y_pred) /
                                  np.clip(np.abs(y_true), 1e-6, None))) * 100)

    z_true   = np.where(yr_true_ratio < 0.9, 0, np.where(yr_true_ratio <= 1.1, 1, 2))
    z_pred   = np.where(yr_pred_ratio < 0.9, 0, np.where(yr_pred_ratio <= 1.1, 1, 2))
    zone_acc = float((z_true == z_pred).mean())
    cm       = confusion_matrix(z_true, z_pred, labels=[0, 1, 2])

    if tag:
        print(f"  {tag:8s} | R²={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}"
              f"  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  ZoneAcc={zone_acc:.4f}")
    return dict(r2=r2, rmse=rmse, mae=mae, nrmse=nrmse,
                mape=mape, zone_acc=zone_acc, cm=cm)


def print_summary(records, title="FINAL SUMMARY"):
    keys = ["r2", "rmse", "mae", "nrmse", "mape", "zone_acc"]
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    for k in keys:
        vals = [r[k] for r in records]
        unit = "%" if k in ("nrmse", "mape") else ""
        print(f"  {k.upper():9s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}{unit}")
    cm_sum = sum(r["cm"] for r in records)
    print(f"\n  Zone Confusion Matrix (sum over folds):")
    print(f"  Rows=true [low/med/high]  Cols=pred")
    for row in cm_sum:
        print("  ", row)
    print("=" * 60)

# ─────────────────────────────────────────────
# 5.  TRAIN ONE FOLD
# ─────────────────────────────────────────────
def train_fold(fold_idx, tr_idx, te_idx, overall_start, completed_folds):
    t_fold = time.time()
    print(f"\n── Fold {fold_idx+1}/{N_SPLITS} "
          f"| train={len(tr_idx):,}  test={len(te_idx):,} ──")

    # Year-mean normalisation (train only — no leakage)
    y_tr_raw    = y_raw[tr_idx]
    yr_tr       = years[tr_idx]
    yr_map      = {int(yr): float(y_tr_raw[yr_tr == yr].mean())
                   for yr in np.unique(yr_tr)}
    global_mean = float(y_tr_raw.mean())

    def _norm(idx):
        mu = np.array([yr_map.get(int(yr), global_mean) for yr in years[idx]],
                      dtype=np.float32)
        return y_raw[idx] / mu, mu

    y_tr_norm, _     = _norm(tr_idx)
    y_te_norm, mu_te = _norm(te_idx)

    # Standardise SAR per-fold (train stats only)
    def _standardize_seq(tr, te):
        flat = tr.reshape(-1, tr.shape[-1])
        mu, sd = flat.mean(0), flat.std(0) + 1e-6
        return (tr - mu) / sd, (te - mu) / sd

    seq_tr_std, seq_te_std = _standardize_seq(
        X_seq_filled[tr_idx], X_seq_filled[te_idx])

    # Standardise static features (train stats only)
    s_mu = X_static[tr_idx].mean(0);  s_sd = X_static[tr_idx].std(0) + 1e-6
    stat_tr = (X_static[tr_idx] - s_mu) / s_sd
    stat_te = (X_static[te_idx] - s_mu) / s_sd

    # DataLoaders
    def _loader(seq, stat, y_norm, shuffle):
        ds = TensorDataset(
            torch.from_numpy(seq),
            torch.from_numpy(stat),
            torch.from_numpy(y_norm),
        )
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

    tr_loader = _loader(seq_tr_std, stat_tr, y_tr_norm, shuffle=True)
    te_loader = _loader(seq_te_std, stat_te, y_te_norm, shuffle=False)

    # Model — FP32 only
    model     = RNNRegressor(seq_dim=X_seq_filled.shape[2],
                              stat_dim=len(STATIC_COLS),
                              hidden=HIDDEN).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=N_EPOCHS)
    loss_fn   = nn.SmoothL1Loss()

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    t_train        = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        epoch_loss = 0.0
        for xs, xst, yb in tr_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE,  non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xs, xst), yb)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()

        # ── validate ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, xst, yb in te_loader:
                xs  = xs.to(DEVICE,  non_blocking=True)
                xst = xst.to(DEVICE, non_blocking=True)
                yb  = yb.to(DEVICE,  non_blocking=True)
                val_loss += loss_fn(model(xs, xst), yb).item()

        scheduler.step()

        # ── early stopping ──
        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        # ETA calculation
        elapsed  = time.time() - overall_start
        progress = (completed_folds + epoch / N_EPOCHS) / N_SPLITS
        eta      = elapsed * (1 - progress) / max(progress, 1e-6)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | "
                  f"train={epoch_loss/len(tr_loader):.4f}  "
                  f"val={val_loss/len(te_loader):.4f}  "
                  f"patience={patience_count}/{PATIENCE}  "
                  f"elapsed={fmt_time(time.time()-t_train)}  "
                  f"ETA≈{fmt_time(eta)}")

        if patience_count >= PATIENCE:
            print(f"  ⏹  Early stop at epoch {epoch}  |  "
                  f"elapsed={fmt_time(time.time()-t_train)}")
            break

    # ── Inference with best weights ──
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    model.eval()
    preds_ratio = []
    with torch.no_grad():
        for xs, xst, _ in te_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            preds_ratio.append(model(xs, xst).cpu().numpy())

    p_ratio  = np.concatenate(preds_ratio)
    p_raw    = p_ratio * mu_te
    y_te_raw = y_raw[te_idx]

    metrics = all_metrics(
        y_true=y_te_raw, y_pred=p_raw,
        yr_true_ratio=y_te_norm,
        yr_pred_ratio=p_ratio,
        tag=f"Fold {fold_idx+1}",
    )
    print(f"  Fold wall time: {fmt_time(time.time()-t_fold)}")
    return metrics

# ─────────────────────────────────────────────
# 6.  MAIN CV LOOP
# ─────────────────────────────────────────────
all_results = []
for i, (tr_idx, te_idx) in enumerate(folds):
    all_results.append(train_fold(i, tr_idx, te_idx, t0_total, i))

print_summary(all_results, title="RNN — 5-Fold CV Summary")
print(f"\nTotal wall time: {fmt_time(time.time()-t0_total)}")

RNN Yield Regressor  —  FP32, no AMP
Device : cuda  (NVIDIA H200)
Loaded  : 2,108,996 rows | 349 cols  (dropped 22 cols)
X_seq   : (2108996, 152, 4)  (T=152 steps, 4 ch incl. NaN mask)
X_static: (2108996, 2)
NaNs after fill: 0
Folds   : 5  (GroupKFold on 'farm_name')


── Fold 1/5 | train=1,687,134  test=421,862 ──
  Epoch 001 | train=0.0779  val=0.0277  patience=0/5  elapsed=00m 08s  ETA≈1h 48m 40s
  Epoch 005 | train=0.0264  val=0.0261  patience=0/5  elapsed=00m 22s  ETA≈28m 02s
  Epoch 010 | train=0.0257  val=0.0252  patience=0/5  elapsed=00m 40s  ETA≈17m 44s
  Epoch 015 | train=0.0253  val=0.0244  patience=0/5  elapsed=00m 58s  ETA≈14m 01s
  Epoch 020 | train=0.0252  val=0.0240  patience=0/5  elapsed=01m 15s  ETA≈12m 02s
  Epoch 025 | train=0.0250  val=0.0238  patience=1/5  elapsed=01m 33s  ETA≈10m 42s
  Epoch 030 | train=0.0250  val=0.0238  patience=4/5  elapsed=01m 51s  ETA≈09m 45s
  Fold 1   | R²=0.2659  RMSE=40.19  MAE=29.80  NRMSE=13.43%  MAPE=22.76%  ZoneAcc=0.4978
  Fold wal

# Wheat

In [4]:
"""
Optimized RNN Yield Regressor  (FP32 — no AMP)
- Mirrors the LSTM script exactly: same data pipeline, metrics, fold structure
- FP32 throughout (avoids cuBLAS FP16 issues)
- Persistent DataLoader workers + pinned memory
- Early stopping (patience=5)
- Cosine LR annealing
- All metrics: R², RMSE, MAE, NRMSE, MAPE, Zone Accuracy + confusion matrix
- Per-fold + aggregate timing with ETA
"""

import re, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  CONFIG  ← only section you need to edit
# ─────────────────────────────────────────────
DATA_PATH   = "../data/wheat_2016_2023.parquet"   # swap crop here
TARGET_COL  = "yield"
GROUP_COL   = "farm_name"
N_SPLITS    = 5
N_EPOCHS    = 30         # RNN converges faster than LSTM; 30 is a safe ceiling
PATIENCE    = 5          # stop early if val loss plateaus
BATCH_SIZE  = 16_384
LR          = 3e-4       # slightly lower than LSTM — RNN gradients can be noisy
HIDDEN      = 64         # RNN is simpler than LSTM; 64 is plenty
NUM_WORKERS = 8
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────
# 1.  LOAD & CLEAN
# ─────────────────────────────────────────────
t0_total = time.time()
print("=" * 60)
print("RNN Yield Regressor  —  FP32, no AMP")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})"
      if torch.cuda.is_available() else ""))
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df = df[df[TARGET_COL].notna()].copy()

drop_cols  = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]
drop_cols += [c for c in df.columns
              if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]
df.drop(columns=drop_cols, errors="ignore", inplace=True)
print(f"Loaded  : {len(df):,} rows | {len(df.columns)} cols  "
      f"(dropped {len(drop_cols)} cols)")

# ─────────────────────────────────────────────
# 2.  BUILD FEATURES
# ─────────────────────────────────────────────
def _suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_map = {_suffix(c): c for c in df.columns if c.startswith("VV_") and _suffix(c)}
vh_map = {_suffix(c): c for c in df.columns if c.startswith("VH_") and _suffix(c)}
common = sorted(set(vv_map) & set(vh_map))
T      = len(common)

vv_arr = df[[vv_map[s] for s in common]].to_numpy(dtype=np.float32)
vh_arr = df[[vh_map[s] for s in common]].to_numpy(dtype=np.float32)
X_seq  = np.stack([vv_arr, vh_arr], axis=2)                              # (N, T, 2)

nan_mask     = np.isnan(X_seq).astype(np.float32)
X_seq_filled = np.concatenate([np.nan_to_num(X_seq), nan_mask], axis=2)  # (N, T, 4)

STATIC_COLS = ["Year", "latitude"]
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)

y_raw  = df[TARGET_COL].to_numpy(dtype=np.float32)
groups = df[GROUP_COL].astype(str).to_numpy()
years  = df["Year"].to_numpy(dtype=np.int32)

print(f"X_seq   : {X_seq_filled.shape}  (T={T} steps, 4 ch incl. NaN mask)")
print(f"X_static: {X_static.shape}")
print(f"NaNs after fill: {np.isnan(X_seq_filled).sum()}")

gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y_raw, groups=groups))
print(f"Folds   : {len(folds)}  (GroupKFold on '{GROUP_COL}')\n")

# ─────────────────────────────────────────────
# 3.  MODEL
# ─────────────────────────────────────────────
class RNNRegressor(nn.Module):
    def __init__(self, seq_dim: int, stat_dim: int, hidden: int = 64):
        super().__init__()
        self.rnn = nn.RNN(seq_dim, hidden, num_layers=2,
                          batch_first=True, dropout=0.2, nonlinearity="tanh")
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),       nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        _, h = self.rnn(x_seq)            # h: (num_layers, B, hidden)
        s    = self.stat_mlp(x_stat)
        return self.head(torch.cat([h[-1], s], dim=1)).squeeze(1)

# ─────────────────────────────────────────────
# 4.  METRIC HELPERS
# ─────────────────────────────────────────────
def fmt_time(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m:02d}m {s:02d}s"

def all_metrics(y_true, y_pred, yr_true_ratio, yr_pred_ratio, tag=""):
    r2    = r2_score(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    rng   = float(np.max(y_true) - np.min(y_true))
    nrmse = (rmse / rng * 100) if rng > 0 else 0.0
    mape  = float(np.mean(np.abs((y_true - y_pred) /
                                  np.clip(np.abs(y_true), 1e-6, None))) * 100)

    z_true   = np.where(yr_true_ratio < 0.9, 0, np.where(yr_true_ratio <= 1.1, 1, 2))
    z_pred   = np.where(yr_pred_ratio < 0.9, 0, np.where(yr_pred_ratio <= 1.1, 1, 2))
    zone_acc = float((z_true == z_pred).mean())
    cm       = confusion_matrix(z_true, z_pred, labels=[0, 1, 2])

    if tag:
        print(f"  {tag:8s} | R²={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}"
              f"  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  ZoneAcc={zone_acc:.4f}")
    return dict(r2=r2, rmse=rmse, mae=mae, nrmse=nrmse,
                mape=mape, zone_acc=zone_acc, cm=cm)


def print_summary(records, title="FINAL SUMMARY"):
    keys = ["r2", "rmse", "mae", "nrmse", "mape", "zone_acc"]
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    for k in keys:
        vals = [r[k] for r in records]
        unit = "%" if k in ("nrmse", "mape") else ""
        print(f"  {k.upper():9s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}{unit}")
    cm_sum = sum(r["cm"] for r in records)
    print(f"\n  Zone Confusion Matrix (sum over folds):")
    print(f"  Rows=true [low/med/high]  Cols=pred")
    for row in cm_sum:
        print("  ", row)
    print("=" * 60)

# ─────────────────────────────────────────────
# 5.  TRAIN ONE FOLD
# ─────────────────────────────────────────────
def train_fold(fold_idx, tr_idx, te_idx, overall_start, completed_folds):
    t_fold = time.time()
    print(f"\n── Fold {fold_idx+1}/{N_SPLITS} "
          f"| train={len(tr_idx):,}  test={len(te_idx):,} ──")

    # Year-mean normalisation (train only — no leakage)
    y_tr_raw    = y_raw[tr_idx]
    yr_tr       = years[tr_idx]
    yr_map      = {int(yr): float(y_tr_raw[yr_tr == yr].mean())
                   for yr in np.unique(yr_tr)}
    global_mean = float(y_tr_raw.mean())

    def _norm(idx):
        mu = np.array([yr_map.get(int(yr), global_mean) for yr in years[idx]],
                      dtype=np.float32)
        return y_raw[idx] / mu, mu

    y_tr_norm, _     = _norm(tr_idx)
    y_te_norm, mu_te = _norm(te_idx)

    # Standardise SAR per-fold (train stats only)
    def _standardize_seq(tr, te):
        flat = tr.reshape(-1, tr.shape[-1])
        mu, sd = flat.mean(0), flat.std(0) + 1e-6
        return (tr - mu) / sd, (te - mu) / sd

    seq_tr_std, seq_te_std = _standardize_seq(
        X_seq_filled[tr_idx], X_seq_filled[te_idx])

    # Standardise static features (train stats only)
    s_mu = X_static[tr_idx].mean(0);  s_sd = X_static[tr_idx].std(0) + 1e-6
    stat_tr = (X_static[tr_idx] - s_mu) / s_sd
    stat_te = (X_static[te_idx] - s_mu) / s_sd

    # DataLoaders
    def _loader(seq, stat, y_norm, shuffle):
        ds = TensorDataset(
            torch.from_numpy(seq),
            torch.from_numpy(stat),
            torch.from_numpy(y_norm),
        )
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

    tr_loader = _loader(seq_tr_std, stat_tr, y_tr_norm, shuffle=True)
    te_loader = _loader(seq_te_std, stat_te, y_te_norm, shuffle=False)

    # Model — FP32 only
    model     = RNNRegressor(seq_dim=X_seq_filled.shape[2],
                              stat_dim=len(STATIC_COLS),
                              hidden=HIDDEN).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=N_EPOCHS)
    loss_fn   = nn.SmoothL1Loss()

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    t_train        = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        epoch_loss = 0.0
        for xs, xst, yb in tr_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE,  non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xs, xst), yb)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()

        # ── validate ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, xst, yb in te_loader:
                xs  = xs.to(DEVICE,  non_blocking=True)
                xst = xst.to(DEVICE, non_blocking=True)
                yb  = yb.to(DEVICE,  non_blocking=True)
                val_loss += loss_fn(model(xs, xst), yb).item()

        scheduler.step()

        # ── early stopping ──
        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        # ETA calculation
        elapsed  = time.time() - overall_start
        progress = (completed_folds + epoch / N_EPOCHS) / N_SPLITS
        eta      = elapsed * (1 - progress) / max(progress, 1e-6)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | "
                  f"train={epoch_loss/len(tr_loader):.4f}  "
                  f"val={val_loss/len(te_loader):.4f}  "
                  f"patience={patience_count}/{PATIENCE}  "
                  f"elapsed={fmt_time(time.time()-t_train)}  "
                  f"ETA≈{fmt_time(eta)}")

        if patience_count >= PATIENCE:
            print(f"  ⏹  Early stop at epoch {epoch}  |  "
                  f"elapsed={fmt_time(time.time()-t_train)}")
            break

    # ── Inference with best weights ──
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    model.eval()
    preds_ratio = []
    with torch.no_grad():
        for xs, xst, _ in te_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            preds_ratio.append(model(xs, xst).cpu().numpy())

    p_ratio  = np.concatenate(preds_ratio)
    p_raw    = p_ratio * mu_te
    y_te_raw = y_raw[te_idx]

    metrics = all_metrics(
        y_true=y_te_raw, y_pred=p_raw,
        yr_true_ratio=y_te_norm,
        yr_pred_ratio=p_ratio,
        tag=f"Fold {fold_idx+1}",
    )
    print(f"  Fold wall time: {fmt_time(time.time()-t_fold)}")
    return metrics

# ─────────────────────────────────────────────
# 6.  MAIN CV LOOP
# ─────────────────────────────────────────────
all_results = []
for i, (tr_idx, te_idx) in enumerate(folds):
    all_results.append(train_fold(i, tr_idx, te_idx, t0_total, i))

print_summary(all_results, title="RNN — 5-Fold CV Summary")
print(f"\nTotal wall time: {fmt_time(time.time()-t0_total)}")

RNN Yield Regressor  —  FP32, no AMP
Device : cuda  (NVIDIA H200)
Loaded  : 495,789 rows | 349 cols  (dropped 22 cols)
X_seq   : (495789, 152, 4)  (T=152 steps, 4 ch incl. NaN mask)
X_static: (495789, 2)
NaNs after fill: 0
Folds   : 5  (GroupKFold on 'farm_name')


── Fold 1/5 | train=396,645  test=99,144 ──
  Epoch 001 | train=0.3725  val=0.1148  patience=0/5  elapsed=00m 04s  ETA≈29m 27s
  Epoch 005 | train=0.0360  val=0.0218  patience=0/5  elapsed=00m 08s  ETA≈07m 47s
  Epoch 010 | train=0.0349  val=0.0218  patience=5/5  elapsed=00m 14s  ETA≈05m 04s
  ⏹  Early stop at epoch 10  |  elapsed=00m 14s
  Fold 1   | R²=0.3354  RMSE=17.16  MAE=13.34  NRMSE=10.40%  MAPE=20.38%  ZoneAcc=0.3706
  Fold wall time: 00m 19s

── Fold 2/5 | train=396,657  test=99,132 ──
  Epoch 001 | train=0.2041  val=0.0442  patience=0/5  elapsed=00m 04s  ETA≈02m 02s
  Epoch 005 | train=0.0314  val=0.0360  patience=0/5  elapsed=00m 09s  ETA≈01m 59s
  Epoch 010 | train=0.0303  val=0.0329  patience=0/5  elapsed=00m 1

# Soybeans

In [5]:
"""
Optimized RNN Yield Regressor  (FP32 — no AMP)
- Mirrors the LSTM script exactly: same data pipeline, metrics, fold structure
- FP32 throughout (avoids cuBLAS FP16 issues)
- Persistent DataLoader workers + pinned memory
- Early stopping (patience=5)
- Cosine LR annealing
- All metrics: R², RMSE, MAE, NRMSE, MAPE, Zone Accuracy + confusion matrix
- Per-fold + aggregate timing with ETA
"""

import re, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0.  CONFIG  ← only section you need to edit
# ─────────────────────────────────────────────
DATA_PATH   = "../data/soybeans_2016_2023.parquet"   # swap crop here
TARGET_COL  = "yield"
GROUP_COL   = "farm_name"
N_SPLITS    = 5
N_EPOCHS    = 30         # RNN converges faster than LSTM; 30 is a safe ceiling
PATIENCE    = 5          # stop early if val loss plateaus
BATCH_SIZE  = 16_384
LR          = 3e-4       # slightly lower than LSTM — RNN gradients can be noisy
HIDDEN      = 64         # RNN is simpler than LSTM; 64 is plenty
NUM_WORKERS = 8
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────
# 1.  LOAD & CLEAN
# ─────────────────────────────────────────────
t0_total = time.time()
print("=" * 60)
print("RNN Yield Regressor  —  FP32, no AMP")
print(f"Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})"
      if torch.cuda.is_available() else ""))
print("=" * 60)

df = pd.read_parquet(DATA_PATH)
df = df[df[TARGET_COL].notna()].copy()

drop_cols  = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]
drop_cols += [c for c in df.columns
              if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]
df.drop(columns=drop_cols, errors="ignore", inplace=True)
print(f"Loaded  : {len(df):,} rows | {len(df.columns)} cols  "
      f"(dropped {len(drop_cols)} cols)")

# ─────────────────────────────────────────────
# 2.  BUILD FEATURES
# ─────────────────────────────────────────────
def _suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_map = {_suffix(c): c for c in df.columns if c.startswith("VV_") and _suffix(c)}
vh_map = {_suffix(c): c for c in df.columns if c.startswith("VH_") and _suffix(c)}
common = sorted(set(vv_map) & set(vh_map))
T      = len(common)

vv_arr = df[[vv_map[s] for s in common]].to_numpy(dtype=np.float32)
vh_arr = df[[vh_map[s] for s in common]].to_numpy(dtype=np.float32)
X_seq  = np.stack([vv_arr, vh_arr], axis=2)                              # (N, T, 2)

nan_mask     = np.isnan(X_seq).astype(np.float32)
X_seq_filled = np.concatenate([np.nan_to_num(X_seq), nan_mask], axis=2)  # (N, T, 4)

STATIC_COLS = ["Year", "latitude"]
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)

y_raw  = df[TARGET_COL].to_numpy(dtype=np.float32)
groups = df[GROUP_COL].astype(str).to_numpy()
years  = df["Year"].to_numpy(dtype=np.int32)

print(f"X_seq   : {X_seq_filled.shape}  (T={T} steps, 4 ch incl. NaN mask)")
print(f"X_static: {X_static.shape}")
print(f"NaNs after fill: {np.isnan(X_seq_filled).sum()}")

gkf   = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y_raw, groups=groups))
print(f"Folds   : {len(folds)}  (GroupKFold on '{GROUP_COL}')\n")

# ─────────────────────────────────────────────
# 3.  MODEL
# ─────────────────────────────────────────────
class RNNRegressor(nn.Module):
    def __init__(self, seq_dim: int, stat_dim: int, hidden: int = 64):
        super().__init__()
        self.rnn = nn.RNN(seq_dim, hidden, num_layers=2,
                          batch_first=True, dropout=0.2, nonlinearity="tanh")
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32), nn.ReLU(),
            nn.Linear(32, 32),       nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + 32, 64), nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq: torch.Tensor, x_stat: torch.Tensor) -> torch.Tensor:
        _, h = self.rnn(x_seq)            # h: (num_layers, B, hidden)
        s    = self.stat_mlp(x_stat)
        return self.head(torch.cat([h[-1], s], dim=1)).squeeze(1)

# ─────────────────────────────────────────────
# 4.  METRIC HELPERS
# ─────────────────────────────────────────────
def fmt_time(sec):
    sec = max(0.0, float(sec))
    h, rem = divmod(int(sec), 3600)
    m, s   = divmod(rem, 60)
    return f"{h}h {m:02d}m {s:02d}s" if h else f"{m:02d}m {s:02d}s"

def all_metrics(y_true, y_pred, yr_true_ratio, yr_pred_ratio, tag=""):
    r2    = r2_score(y_true, y_pred)
    rmse  = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae   = float(mean_absolute_error(y_true, y_pred))
    rng   = float(np.max(y_true) - np.min(y_true))
    nrmse = (rmse / rng * 100) if rng > 0 else 0.0
    mape  = float(np.mean(np.abs((y_true - y_pred) /
                                  np.clip(np.abs(y_true), 1e-6, None))) * 100)

    z_true   = np.where(yr_true_ratio < 0.9, 0, np.where(yr_true_ratio <= 1.1, 1, 2))
    z_pred   = np.where(yr_pred_ratio < 0.9, 0, np.where(yr_pred_ratio <= 1.1, 1, 2))
    zone_acc = float((z_true == z_pred).mean())
    cm       = confusion_matrix(z_true, z_pred, labels=[0, 1, 2])

    if tag:
        print(f"  {tag:8s} | R²={r2:.4f}  RMSE={rmse:.2f}  MAE={mae:.2f}"
              f"  NRMSE={nrmse:.2f}%  MAPE={mape:.2f}%  ZoneAcc={zone_acc:.4f}")
    return dict(r2=r2, rmse=rmse, mae=mae, nrmse=nrmse,
                mape=mape, zone_acc=zone_acc, cm=cm)


def print_summary(records, title="FINAL SUMMARY"):
    keys = ["r2", "rmse", "mae", "nrmse", "mape", "zone_acc"]
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    for k in keys:
        vals = [r[k] for r in records]
        unit = "%" if k in ("nrmse", "mape") else ""
        print(f"  {k.upper():9s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}{unit}")
    cm_sum = sum(r["cm"] for r in records)
    print(f"\n  Zone Confusion Matrix (sum over folds):")
    print(f"  Rows=true [low/med/high]  Cols=pred")
    for row in cm_sum:
        print("  ", row)
    print("=" * 60)

# ─────────────────────────────────────────────
# 5.  TRAIN ONE FOLD
# ─────────────────────────────────────────────
def train_fold(fold_idx, tr_idx, te_idx, overall_start, completed_folds):
    t_fold = time.time()
    print(f"\n── Fold {fold_idx+1}/{N_SPLITS} "
          f"| train={len(tr_idx):,}  test={len(te_idx):,} ──")

    # Year-mean normalisation (train only — no leakage)
    y_tr_raw    = y_raw[tr_idx]
    yr_tr       = years[tr_idx]
    yr_map      = {int(yr): float(y_tr_raw[yr_tr == yr].mean())
                   for yr in np.unique(yr_tr)}
    global_mean = float(y_tr_raw.mean())

    def _norm(idx):
        mu = np.array([yr_map.get(int(yr), global_mean) for yr in years[idx]],
                      dtype=np.float32)
        return y_raw[idx] / mu, mu

    y_tr_norm, _     = _norm(tr_idx)
    y_te_norm, mu_te = _norm(te_idx)

    # Standardise SAR per-fold (train stats only)
    def _standardize_seq(tr, te):
        flat = tr.reshape(-1, tr.shape[-1])
        mu, sd = flat.mean(0), flat.std(0) + 1e-6
        return (tr - mu) / sd, (te - mu) / sd

    seq_tr_std, seq_te_std = _standardize_seq(
        X_seq_filled[tr_idx], X_seq_filled[te_idx])

    # Standardise static features (train stats only)
    s_mu = X_static[tr_idx].mean(0);  s_sd = X_static[tr_idx].std(0) + 1e-6
    stat_tr = (X_static[tr_idx] - s_mu) / s_sd
    stat_te = (X_static[te_idx] - s_mu) / s_sd

    # DataLoaders
    def _loader(seq, stat, y_norm, shuffle):
        ds = TensorDataset(
            torch.from_numpy(seq),
            torch.from_numpy(stat),
            torch.from_numpy(y_norm),
        )
        return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=(NUM_WORKERS > 0))

    tr_loader = _loader(seq_tr_std, stat_tr, y_tr_norm, shuffle=True)
    te_loader = _loader(seq_te_std, stat_te, y_te_norm, shuffle=False)

    # Model — FP32 only
    model     = RNNRegressor(seq_dim=X_seq_filled.shape[2],
                              stat_dim=len(STATIC_COLS),
                              hidden=HIDDEN).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=N_EPOCHS)
    loss_fn   = nn.SmoothL1Loss()

    best_val_loss  = float("inf")
    best_state     = None
    patience_count = 0
    t_train        = time.time()

    for epoch in range(1, N_EPOCHS + 1):
        # ── train ──
        model.train()
        epoch_loss = 0.0
        for xs, xst, yb in tr_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            yb  = yb.to(DEVICE,  non_blocking=True)
            optimiser.zero_grad(set_to_none=True)
            loss = loss_fn(model(xs, xst), yb)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()

        # ── validate ──
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xs, xst, yb in te_loader:
                xs  = xs.to(DEVICE,  non_blocking=True)
                xst = xst.to(DEVICE, non_blocking=True)
                yb  = yb.to(DEVICE,  non_blocking=True)
                val_loss += loss_fn(model(xs, xst), yb).item()

        scheduler.step()

        # ── early stopping ──
        if val_loss < best_val_loss - 1e-5:
            best_val_loss  = val_loss
            best_state     = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        # ETA calculation
        elapsed  = time.time() - overall_start
        progress = (completed_folds + epoch / N_EPOCHS) / N_SPLITS
        eta      = elapsed * (1 - progress) / max(progress, 1e-6)

        if epoch % 5 == 0 or epoch == 1:
            print(f"  Epoch {epoch:03d} | "
                  f"train={epoch_loss/len(tr_loader):.4f}  "
                  f"val={val_loss/len(te_loader):.4f}  "
                  f"patience={patience_count}/{PATIENCE}  "
                  f"elapsed={fmt_time(time.time()-t_train)}  "
                  f"ETA≈{fmt_time(eta)}")

        if patience_count >= PATIENCE:
            print(f"  ⏹  Early stop at epoch {epoch}  |  "
                  f"elapsed={fmt_time(time.time()-t_train)}")
            break

    # ── Inference with best weights ──
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    model.eval()
    preds_ratio = []
    with torch.no_grad():
        for xs, xst, _ in te_loader:
            xs  = xs.to(DEVICE,  non_blocking=True)
            xst = xst.to(DEVICE, non_blocking=True)
            preds_ratio.append(model(xs, xst).cpu().numpy())

    p_ratio  = np.concatenate(preds_ratio)
    p_raw    = p_ratio * mu_te
    y_te_raw = y_raw[te_idx]

    metrics = all_metrics(
        y_true=y_te_raw, y_pred=p_raw,
        yr_true_ratio=y_te_norm,
        yr_pred_ratio=p_ratio,
        tag=f"Fold {fold_idx+1}",
    )
    print(f"  Fold wall time: {fmt_time(time.time()-t_fold)}")
    return metrics

# ─────────────────────────────────────────────
# 6.  MAIN CV LOOP
# ─────────────────────────────────────────────
all_results = []
for i, (tr_idx, te_idx) in enumerate(folds):
    all_results.append(train_fold(i, tr_idx, te_idx, t0_total, i))

print_summary(all_results, title="RNN — 5-Fold CV Summary")
print(f"\nTotal wall time: {fmt_time(time.time()-t0_total)}")

RNN Yield Regressor  —  FP32, no AMP
Device : cuda  (NVIDIA H200)
Loaded  : 694,250 rows | 349 cols  (dropped 22 cols)
X_seq   : (694250, 152, 4)  (T=152 steps, 4 ch incl. NaN mask)
X_static: (694250, 2)
NaNs after fill: 0
Folds   : 5  (GroupKFold on 'farm_name')


── Fold 1/5 | train=555,477  test=138,773 ──
  Epoch 001 | train=0.1902  val=0.0563  patience=0/5  elapsed=00m 04s  ETA≈39m 29s
  Epoch 005 | train=0.0316  val=0.0273  patience=0/5  elapsed=00m 10s  ETA≈10m 19s
  Epoch 010 | train=0.0292  val=0.0252  patience=0/5  elapsed=00m 17s  ETA≈06m 38s
  Epoch 015 | train=0.0276  val=0.0246  patience=0/5  elapsed=00m 24s  ETA≈05m 21s
  Epoch 020 | train=0.0267  val=0.0242  patience=0/5  elapsed=00m 31s  ETA≈04m 38s
  Epoch 025 | train=0.0263  val=0.0241  patience=2/5  elapsed=00m 39s  ETA≈04m 10s
  ⏹  Early stop at epoch 28  |  elapsed=00m 43s
  Fold 1   | R²=0.3217  RMSE=10.98  MAE=8.38  NRMSE=11.15%  MAPE=22.85%  ZoneAcc=0.4673
  Fold wall time: 00m 49s

── Fold 2/5 | train=555,403 